# STEM-EBIC / SEEBIC Simulation on TEM Samples

Pipeline that turns a TEM cross-section image + SIMS doping profiles + material
property table into a full EBIC / SEEBIC simulation:

1. Load image, auto-detect the scalebar, and derive pixel size in nm.
2. Auto-segment the image into regions (scikit-image), then map each region to
   a `Material_name` from the material table (or the keyword `SIMS`).
3. Load the material table; BGN, Arora mobility, and E_F are computed on demand.
4. Load SIMS N/P profiles, interpolate, and extend with the substrate
   concentration for depths past the last sample.
5. Build 2-D doping maps Na(x,y), Nd(x,y), N_net(x,y). SIMS regions take the
   top edge of their bounding box as y = 0 (surface).
6. Electron beam: Kanaya-Okayama range + ellipsoidal generation bulb clipped to
   the TEM foil thickness (Z = 100 nm default).
7. Physics: E-field via Poisson integration (no averaging), depletion width,
   diffusion length (Arora + SRH), and collection probability with drift.
8. Circuit: 3 terminal types (ammeter+GND, voltage, GND). Contact type resolved
   from work-function differences.
9. Band diagram along the active contact-to-contact path.
10. EBIC and SEEBIC 2-D maps.


## 0. Imports and constants

Install dependencies if needed by uncommenting the `%pip` line.

In [ ]:
# %pip install -q numpy scipy pandas matplotlib pillow tifffile scikit-image scikit-learn pyyaml pytesseract

import math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from matplotlib.patches import Rectangle

from PIL import Image

from scipy.interpolate import interp1d
from scipy.integrate import cumulative_trapezoid
from scipy.ndimage import binary_dilation, distance_transform_edt
from scipy.signal import fftconvolve

from skimage import color, filters, measure, morphology, segmentation
from sklearn.cluster import KMeans

try:
    import pytesseract
    HAVE_TESSERACT = True
except Exception:
    HAVE_TESSERACT = False

# Physical constants
q     = 1.602176634e-19
kB    = 1.380649e-23
h     = 6.62607015e-34
m0    = 9.1093837015e-31
eps0  = 8.8541878128e-12
T_default  = 300.0
Vt_default = kB * T_default / q

print("imports OK; pytesseract:", HAVE_TESSERACT)


## 1. Configuration

Edit the cell below to point at your data files, beam settings, circuit
terminals, substrate doping, and the region → material mapping. The mapping
is filled in after the segmentation step (section 3) the first time; re-run
this cell with the final assignment afterward.

In [ ]:
CONFIG = {
    "image_path": "image.tif",
    "material_table_path": "Material_table_for_ebic_cal.csv",
    "sims_n_path": "SIMSNdata.csv",
    "sims_p_path": "SIMSPdata.csv",

    "thickness_nm": 100.0,

    # Leave None to auto-detect from scalebar. scalebar_nominal_nm is the
    # physical length of the rendered bar; fallback if OCR fails.
    "pixel_size_nm": None,
    "scalebar_nominal_nm": 500.0,
    "scalebar_search_frac": 0.25,

    "grid_dx_nm": None,   # None -> equal to pixel_size_nm

    "segmentation": {
        "method": "kmeans_felzenszwalb",
        "n_clusters": 4,
        "felzenszwalb_scale": 200,
        "felzenszwalb_sigma": 0.8,
        "min_region_px": 200,
    },

    # Fill after viewing the overlay in section 3.
    "region_assignment": {
        # 1: "Al",
        # 2: "SIMS",
        # 3: "P-type_Si",
    },

    "substrate": {
        "type": "P",
        "concentration_cm3": 1e15,
        "blend_nm": 50.0,
    },

    "beam": {
        "energy_keV": 30.0,
        "current_A": 100e-12,
        "backscatter_yield": 0.15,
        "ehp_energy_eV": 3.6,
        "secondary_escape_nm": 2.5,
    },

    "circuit": [
        # {"role": "ammeter_gnd", "region_id": 1, "bias_V": 0.0},
        # {"role": "voltage",     "region_id": 3, "bias_V": 1.0},
    ],

    "temperature_K": 300.0,
}


## 2. Image loading and automatic scalebar detection

Loads the TIFF, converts to RGB, and looks for a horizontal bar in the
bottom `search_frac` of the image (brightest or darkest, aspect ratio ≥ 10:1).
Combines the bar's pixel length with the nominal scalebar value (from OCR
if available, otherwise `CONFIG['scalebar_nominal_nm']`) to derive
`pixel_size_nm`.

In [ ]:
def load_image_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"))

def detect_scalebar(rgb, search_frac=0.25, min_aspect=10.0, ocr=False):
    """Find the longest horizontal bar in the bottom strip."""
    h_img, w_img = rgb.shape[:2]
    y0 = int(h_img * (1.0 - search_frac))
    strip = rgb[y0:]
    gray = color.rgb2gray(strip)
    best = None
    for invert in (False, True):
        img = 1.0 - gray if invert else gray
        thr = filters.threshold_otsu(img)
        bw = morphology.remove_small_objects(img > thr, min_size=20)
        lbl = measure.label(bw)
        for region in measure.regionprops(lbl):
            minr, minc, maxr, maxc = region.bbox
            w, hh = maxc - minc, maxr - minr
            if hh <= 0 or w < 20 or w / max(hh, 1) < min_aspect:
                continue
            if best is None or w > best[0]:
                best = (w, (minr + y0, minc, maxr + y0, maxc))
    if best is None:
        raise RuntimeError("No scalebar candidate found.")
    _, bbox = best
    pixel_length = bbox[3] - bbox[1]
    text = None
    if ocr and HAVE_TESSERACT:
        top = max(bbox[0] - 40, 0)
        crop = rgb[top:bbox[0], max(bbox[1]-10, 0):bbox[3]+10]
        try:
            text = pytesseract.image_to_string(crop).strip()
        except Exception:
            text = None
    return pixel_length, bbox, text

def parse_scalebar_text(text):
    if not text:
        return None
    import re
    m = re.search(r"([0-9.]+)\s*(nm|um|µm|μm|mm)", text.lower())
    if not m:
        return None
    val = float(m.group(1)); unit = m.group(2)
    if unit == "nm": return val
    if unit in ("um", "µm", "μm"): return val * 1000.0
    if unit == "mm": return val * 1e6
    return None

IMG_RGB = load_image_rgb(CONFIG["image_path"])
print("image shape:", IMG_RGB.shape)

bar_bbox = None
if CONFIG["pixel_size_nm"] is None:
    try:
        px_len, bar_bbox, ocr_text = detect_scalebar(
            IMG_RGB, search_frac=CONFIG["scalebar_search_frac"], ocr=True,
        )
        nominal = parse_scalebar_text(ocr_text) or CONFIG["scalebar_nominal_nm"]
        if nominal is None:
            raise RuntimeError("No scalebar nominal; set CONFIG['scalebar_nominal_nm'].")
        CONFIG["pixel_size_nm"] = nominal / px_len
        print(f"Scalebar: {px_len} px -> {nominal} nm  =>  "
              f"pixel_size = {CONFIG['pixel_size_nm']:.3f} nm/px")
        print("OCR text:", repr(ocr_text))
    except Exception as exc:
        print("Auto scalebar detection failed:", exc)
        print("Set CONFIG['pixel_size_nm'] manually and re-run.")

if CONFIG["grid_dx_nm"] is None:
    CONFIG["grid_dx_nm"] = CONFIG["pixel_size_nm"] or 1.0

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(IMG_RGB)
if bar_bbox is not None:
    minr, minc, maxr, maxc = bar_bbox
    ax.add_patch(Rectangle((minc, minr), maxc-minc, maxr-minr,
                           fill=False, edgecolor="red", linewidth=1.5))
    ax.set_title(f"scalebar {maxc-minc} px  |  {CONFIG['pixel_size_nm']:.3f} nm/px")
ax.set_axis_off()
plt.show()


## 3. Automatic region segmentation

K-means on CIE-LAB color to group pixels by contrast, then refined with
Felzenszwalb (or watershed on the Sobel gradient) to split touching
components. Small components are dropped. The overlay annotates every
region with its integer ID so you can fill `CONFIG['region_assignment']`.

In [ ]:
def segment_image(rgb, cfg):
    lab = color.rgb2lab(rgb)
    flat = lab.reshape(-1, 3)
    km = KMeans(n_clusters=cfg["n_clusters"], n_init=5, random_state=0)
    km.fit(flat)
    cluster_map = km.labels_.reshape(rgb.shape[:2])

    if cfg["method"] == "watershed":
        grad = filters.sobel(color.rgb2gray(rgb))
        markers = measure.label(cluster_map)
        seg = segmentation.watershed(grad, markers)
    else:
        fz = segmentation.felzenszwalb(
            rgb, scale=cfg["felzenszwalb_scale"],
            sigma=cfg["felzenszwalb_sigma"], min_size=cfg["min_region_px"],
        )
        combo = cluster_map * (fz.max() + 1) + fz
        seg = measure.label(combo)

    labels = measure.label(seg)
    for reg in measure.regionprops(labels):
        if reg.area < cfg["min_region_px"]:
            minr, minc, maxr, maxc = reg.bbox
            patch = labels[max(minr-1,0):maxr+1, max(minc-1,0):maxc+1]
            neigh = patch[patch != reg.label]
            if neigh.size:
                labels[labels == reg.label] = np.bincount(neigh.ravel()).argmax()
    labels, _, _ = segmentation.relabel_sequential(labels, offset=1)
    return labels

REGIONS = segment_image(IMG_RGB, CONFIG["segmentation"])
n_regions = REGIONS.max()
print(f"Segmented into {n_regions} regions")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(IMG_RGB); axes[0].set_title("input"); axes[0].set_axis_off()
overlay = color.label2rgb(REGIONS, IMG_RGB, alpha=0.45, bg_label=0)
axes[1].imshow(overlay)
for reg in measure.regionprops(REGIONS):
    cy, cx = reg.centroid
    axes[1].text(cx, cy, str(reg.label),
                 color="white", fontsize=10, weight="bold",
                 ha="center", va="center",
                 bbox=dict(facecolor="black", alpha=0.6, pad=1.5))
axes[1].set_title(f"{n_regions} regions"); axes[1].set_axis_off()
plt.tight_layout(); plt.show()
print("=> Fill CONFIG['region_assignment'] with {region_id: material_name, ...},")
print("   then re-run the CONFIG cell (section 1) and this cell.")


## 4. Material table loader with BGN, Arora, and E_F

The CSV is column-oriented (materials along columns, properties down rows).
Semiconductor rows flagged `"Cal"` compute their work function and bandgap
from the local doping using bandgap narrowing (applied when
`|N| >= 1e18 cm^-3`) and charge neutrality for E_F. Arora 1982
coefficients drive the low-field mobility; diffusion length combines that
with an SRH minority lifetime.

In [ ]:
# Arora 1982 mobility parameters for Si at 300 K
AROROA_SI = {
    "electron": dict(mu_min=88.0, mu_0=1252.0, N_ref=1.26e17, alpha=0.88),
    "hole":     dict(mu_min=54.3, mu_0=407.0,  N_ref=2.35e17, alpha=0.88),
}
# SRH minority lifetime
TAU_SI = {
    "electron": dict(tau0=1e-5, N_srh=5e16),
    "hole":     dict(tau0=3e-6, N_srh=5e16),
}

def bgn_delta_eV(N):
    """Slotboom-style BGN, applied when |N| >= 1e18 cm^-3."""
    N = np.asarray(N, dtype=float)
    out = np.zeros_like(N)
    mask = np.abs(N) >= 1e18
    x = np.log(np.clip(np.abs(N[mask]), 1e10, None) / 1e17)
    out[mask] = 9e-3 * (x + np.sqrt(x*x + 0.5))
    return out

def arora_mobility(N, carrier):
    p = AROROA_SI[carrier]
    N = np.clip(np.abs(np.asarray(N, dtype=float)), 1.0, None)
    return p["mu_min"] + p["mu_0"] / (1.0 + (N / p["N_ref"]) ** p["alpha"])

def srh_lifetime(N, carrier):
    p = TAU_SI[carrier]
    N = np.clip(np.abs(np.asarray(N, dtype=float)), 1.0, None)
    return p["tau0"] / (1.0 + N / p["N_srh"])

def diffusion_length(N, carrier, T=T_default):
    mu = arora_mobility(N, carrier) * 1e-4  # -> m^2/V/s
    D  = (kB * T / q) * mu
    tau = srh_lifetime(N, carrier)
    return np.sqrt(D * tau) * 1e9  # nm


@dataclass
class Material:
    name: str
    kind: str
    work_function_eV: object = None
    electron_affinity_eV: float = None
    bandgap_eV: object = None
    rel_permittivity: float = None
    m_eff_e: float = None
    m_eff_h: float = None
    doping_type: str = None

    def Nc(self, T=T_default):
        return 2 * (2*math.pi*self.m_eff_e*m0*kB*T / h**2) ** 1.5 * 1e-6

    def Nv(self, T=T_default):
        return 2 * (2*math.pi*self.m_eff_h*m0*kB*T / h**2) ** 1.5 * 1e-6

    def bandgap(self, N=0.0):
        Eg0 = float(self.bandgap_eV) if self.bandgap_eV not in (None, "Cal") else 1.12
        return Eg0 - bgn_delta_eV(np.asarray(N, dtype=float))

    def work_function(self, N=0.0, T=T_default):
        if self.work_function_eV not in (None, "Cal"):
            return float(self.work_function_eV)
        Eg = self.bandgap(N=N)
        Vt = kB*T/q
        Nc = self.Nc(T)
        if self.doping_type == "N":
            Ec_EF = Vt * np.log(Nc / np.clip(N, 1.0, None))
        elif self.doping_type == "P":
            Nv = self.Nv(T)
            Ec_EF = Eg - Vt * np.log(Nv / np.clip(N, 1.0, None))
        else:
            Ec_EF = Eg / 2
        return self.electron_affinity_eV + Ec_EF


def load_material_table(path):
    raw = pd.read_csv(path, header=None)
    props = raw.iloc[:, 0].astype(str).str.strip().tolist()
    mat_cols = raw.iloc[:, 2:]
    name_row = mat_cols.iloc[1].astype(str).str.strip().tolist()
    type_row = mat_cols.iloc[2].astype(str).str.strip().tolist()

    def numeric(x):
        try:
            return float(str(x).replace(" ", ""))
        except Exception:
            s = str(x).strip()
            return s if s else None

    def row_index(label):
        for i, p in enumerate(props):
            if p.lower().startswith(label.lower()):
                return i
        return None

    idx_wf  = row_index("work_function")
    idx_chi = row_index("electron_affinity")
    idx_eg  = row_index("bandgap")
    idx_eps = row_index("relative_permittivity")
    idx_me  = row_index("effective_e")
    idx_mh  = row_index("effective_h")

    materials = {}
    for j, nm in enumerate(name_row):
        tp = type_row[j]
        def get(idx):
            return numeric(mat_cols.iloc[idx, j]) if idx is not None else None
        doping_type = None
        if nm.lower().startswith("p-type"):
            doping_type = "P"
        elif nm.lower().startswith("n-type"):
            doping_type = "N"
        elif tp.lower() == "semi":
            doping_type = "I"
        materials[nm] = Material(
            name=nm, kind=tp,
            work_function_eV=get(idx_wf),
            electron_affinity_eV=get(idx_chi),
            bandgap_eV=get(idx_eg),
            rel_permittivity=get(idx_eps),
            m_eff_e=get(idx_me), m_eff_h=get(idx_mh),
            doping_type=doping_type,
        )
    return materials

MATERIALS = load_material_table(CONFIG["material_table_path"])
print("Materials loaded:")
for k, m in MATERIALS.items():
    print(f"  {k:12s}  kind={m.kind:9s}  WF={m.work_function_eV}  "
          f"chi={m.electron_affinity_eV}  Eg={m.bandgap_eV}")


## 5. SIMS profile load, interpolation, substrate tail

Cubic interpolation over the sampled range; beyond the last sample, a
`tanh` blend fades into the user-declared substrate concentration so no
spurious field appears at the profile edge.

In [ ]:
def load_sims_csv(path):
    df = pd.read_csv(path, skiprows=[1])
    y = df.iloc[:, 0].astype(float).to_numpy()
    n = df.iloc[:, 1].astype(float).to_numpy()
    order = np.argsort(y)
    return y[order], n[order]

def profile_interpolator(y_nm, N_cm3, substrate_conc, blend_nm=50.0):
    kind = "cubic" if len(y_nm) >= 4 else "linear"
    base = interp1d(y_nm, N_cm3, kind=kind, bounds_error=False,
                    fill_value=(N_cm3[0], N_cm3[-1]))
    y_max = y_nm[-1]
    def f(y):
        y = np.asarray(y, dtype=float)
        N_prof = base(y)
        w = 0.5 * (1 + np.tanh((y - y_max) / max(blend_nm, 1e-3)))
        return (1 - w) * N_prof + w * substrate_conc
    return f

Y_N, Z_N = load_sims_csv(CONFIG["sims_n_path"])
Y_P, Z_P = load_sims_csv(CONFIG["sims_p_path"])
print(f"SIMS N: {len(Y_N)} pts, {Y_N.min():.0f}-{Y_N.max():.0f} nm, peak {Z_N.max():.2e}")
print(f"SIMS P: {len(Y_P)} pts, {Y_P.min():.0f}-{Y_P.max():.0f} nm, peak {Z_P.max():.2e}")

sub_conc = CONFIG["substrate"]["concentration_cm3"]
sub_is_N = CONFIG["substrate"]["type"].upper() == "N"
sub_is_P = CONFIG["substrate"]["type"].upper() == "P"
sims_n = profile_interpolator(Y_N, Z_N, sub_conc if sub_is_N else 0.0,
                              CONFIG["substrate"]["blend_nm"])
sims_p = profile_interpolator(Y_P, Z_P, sub_conc if sub_is_P else 0.0,
                              CONFIG["substrate"]["blend_nm"])

yy = np.linspace(0, max(Y_N.max(), Y_P.max()) + 500, 1200)
fig, ax = plt.subplots(figsize=(6,3.2))
ax.semilogy(yy, sims_n(yy), label="N donors")
ax.semilogy(yy, sims_p(yy), label="P acceptors")
ax.set_xlabel("depth [nm]"); ax.set_ylabel("concentration [cm$^{-3}$]")
ax.set_title(f"SIMS profiles + {CONFIG['substrate']['type']}-substrate tail "
             f"({sub_conc:.0e} cm$^{{-3}}$)")
ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## 6. Doping map construction

`SIMS` regions use their bounding-box top as `y = 0` (surface); depth
grows downward. Both profiles are sampled at every pixel row. The material
per pixel becomes `P-type_Si` or `N-type_Si` from `sign(Nd - Na)`.
Uniform-doped semi regions take a default from `defaults` (per material).

In [ ]:
def build_doping_map(regions, assignment, materials, sims_n, sims_p,
                     pixel_size_nm, defaults=None):
    defaults = defaults or {}
    H, W = regions.shape
    Na = np.zeros((H, W), dtype=float)
    Nd = np.zeros((H, W), dtype=float)
    eff = np.full((H, W), "", dtype=object)
    is_semi = np.zeros((H, W), dtype=bool)

    for reg in measure.regionprops(regions):
        rid = reg.label
        mat_name = assignment.get(rid)
        if mat_name is None:
            continue
        mask = regions == rid
        if mat_name == "SIMS":
            minr, _, _, _ = reg.bbox
            rows, cols = np.where(mask)
            depth_nm = (rows - minr) * pixel_size_nm
            Nd_vals = sims_n(depth_nm)
            Na_vals = sims_p(depth_nm)
            Nd[rows, cols] = Nd_vals
            Na[rows, cols] = Na_vals
            net = Nd_vals - Na_vals
            is_semi[rows, cols] = True
            for r, c, n in zip(rows, cols, net):
                eff[r, c] = "N-type_Si" if n >= 0 else "P-type_Si"
        else:
            mat = materials.get(mat_name)
            if mat is None:
                print(f"  region {rid}: unknown material '{mat_name}'")
                continue
            eff[mask] = mat_name
            if mat.kind == "Semi":
                is_semi |= mask
                N0 = defaults.get(mat_name, 1e15)
                if mat.doping_type == "N":
                    Nd[mask] = N0
                elif mat.doping_type == "P":
                    Na[mask] = N0
    return Na, Nd, Nd - Na, eff, is_semi

if not CONFIG["region_assignment"]:
    print("!! CONFIG['region_assignment'] is empty. View section 3 overlay, "
          "fill the dict, re-run section 1 (config), then this cell.")

Na_map, Nd_map, Nnet_map, EFF_MAT, SEMI_MASK = build_doping_map(
    REGIONS, CONFIG["region_assignment"], MATERIALS, sims_n, sims_p,
    CONFIG["pixel_size_nm"],
    defaults={"P-type_Si": 1e15, "N-type_Si": 1e15, "Si": 1e12},
)
print(f"semi pixels: {SEMI_MASK.sum()} / {SEMI_MASK.size}; "
      f"Nd peak {Nd_map.max():.2e}, Na peak {Na_map.max():.2e}, "
      f"|N_net| peak {np.abs(Nnet_map).max():.2e}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, arr, title in zip(
    axes,
    [np.where(Nd_map > 0, Nd_map, np.nan),
     np.where(Na_map > 0, Na_map, np.nan),
     Nnet_map],
    ["Nd [cm$^{-3}$]", "Na [cm$^{-3}$]", "N_net [cm$^{-3}$]"],
):
    if "net" in title.lower():
        vmax = np.nanmax(np.abs(arr)) or 1.0
        im = ax.imshow(arr, cmap="bwr",
                       norm=SymLogNorm(linthresh=1e14, vmin=-vmax, vmax=vmax))
    else:
        vmax = np.nanmax(arr) if np.isfinite(np.nanmax(arr)) else 1e18
        im = ax.imshow(arr, cmap="viridis",
                       norm=LogNorm(vmin=1e14, vmax=max(vmax, 1e15)))
    ax.set_title(title); ax.set_axis_off()
    plt.colorbar(im, ax=ax, fraction=0.045)
plt.tight_layout(); plt.show()


## 7. Electron beam: Kanaya-Okayama range and generation bulb

$R_{KO}[\mu m] = 0.0276 \cdot A \cdot E^{1.67} / (\rho \cdot Z^{0.889})$

3-D ellipsoidal bulb centered at $0.3 R_{KO}$ depth, integrated along Z
(TEM slab). Total generation:
$G_{tot} = I_{beam} \cdot E_{beam} / q \cdot (1 - \eta_{bs}) / E_{eh}$

In [ ]:
PHYS_MAT_PROPS = {
    "Si": (28.09, 14, 2.33),
    "P-type_Si": (28.09, 14, 2.33),
    "N-type_Si": (28.09, 14, 2.33),
    "Al": (26.98, 13, 2.70),
    "Au": (196.97, 79, 19.30),
    "Pt": (195.08, 78, 21.45),
    "W":  (183.84, 74, 19.25),
}

def kanaya_okayama_nm(material_name, E_keV):
    A, Z, rho = PHYS_MAT_PROPS.get(material_name, (28.09, 14, 2.33))
    return 0.0276 * A * (E_keV ** 1.67) / (rho * (Z ** 0.889)) * 1000.0

def generation_kernel(R_nm, thickness_nm, dx_nm):
    """Return 2-D lateral kernel (integrated over slab depth) + axes."""
    ax_xy = 0.5 * R_nm
    ax_z  = 0.5 * R_nm
    n_xy = max(int(math.ceil(2.5 * ax_xy / dx_nm)), 3)
    n_z  = max(int(math.ceil(thickness_nm / dx_nm)), 1)
    xs = (np.arange(-n_xy, n_xy+1)) * dx_nm
    ys = xs.copy()
    zs = np.linspace(0, thickness_nm, n_z)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")
    z0 = 0.3 * R_nm
    r2 = (X/ax_xy)**2 + (Y/ax_xy)**2 + ((Z - z0)/ax_z)**2
    K = np.exp(-r2 * 1.5)
    if K.sum() > 0:
        K /= K.sum()
    return K.sum(axis=-1), xs, ys

BEAM = CONFIG["beam"]
E_keV = BEAM["energy_keV"]
dominant_semi = None
for rid, mat in CONFIG["region_assignment"].items():
    if mat in MATERIALS and MATERIALS[mat].kind == "Semi":
        dominant_semi = mat; break
dominant_semi = dominant_semi or "Si"
R_KO_nm = kanaya_okayama_nm(dominant_semi, E_keV)
print(f"Kanaya-Okayama in {dominant_semi} @ {E_keV} keV = {R_KO_nm:.1f} nm")

dx_nm = CONFIG["grid_dx_nm"]
G_KERNEL, gx, gy = generation_kernel(R_KO_nm, CONFIG["thickness_nm"], dx_nm)
print("kernel shape:", G_KERNEL.shape, "dx =", dx_nm, "nm")

G_TOT = BEAM["current_A"] / q * (E_keV * 1000.0) * (1.0 - BEAM["backscatter_yield"]) / BEAM["ehp_energy_eV"]
print(f"G_total = {G_TOT:.3e} e-h pairs/s per landing")

fig, ax = plt.subplots(figsize=(4,4))
ax.imshow(G_KERNEL, extent=[gx.min(), gx.max(), gy.max(), gy.min()], cmap="inferno")
ax.set_xlabel("x [nm]"); ax.set_ylabel("y [nm]")
ax.set_title(f"generation kernel (R$_{{KO}}$={R_KO_nm:.0f} nm)")
plt.tight_layout(); plt.show()


## 8. Physics pipeline: E-field, depletion, collection probability

For each column of the sample we form the 1-D `N_net(y)` slice, integrate
Poisson to get E(y), locate junctions (sign flips + high-low steps), and
compute a depletion mask by extending until the integrated potential
matches `V_bi - V_app`. Diffusion length uses the Arora + SRH result;
collection probability is 1 inside depletion, `exp(-d/L)` outside,
with drift enhancement when |E|·L/Vt is large.

In [ ]:
def poisson_efield_1d(N_net_row, eps_r_row, dx_m, E0=0.0):
    rho = q * N_net_row * 1e6           # cm^-3 -> C/m^3
    eps = eps_r_row * eps0
    dE = rho / eps
    E = np.concatenate(([0.0], cumulative_trapezoid(dE, dx=dx_m))) + E0
    return E

def locate_junctions(N_net_row):
    sign = np.sign(N_net_row)
    flips = np.where(np.diff(sign) != 0)[0]
    logN  = np.log10(np.clip(np.abs(N_net_row), 1.0, None))
    grad  = np.abs(np.diff(logN))
    highlow = np.where(grad > 1.0)[0]
    return np.unique(np.concatenate([flips, highlow])).astype(int)

def depletion_mask_1d(N_net_row, eps_r_row, dx_m, junctions, V_bi=0.7, V_app=0.0):
    mask = np.zeros_like(N_net_row, dtype=bool)
    n = len(N_net_row)
    for j in junctions:
        lo, hi = j, j + 1
        for _ in range(200):
            left = max(lo, 0); right = min(hi+1, n)
            seg = N_net_row[left:right]
            eps_seg = eps_r_row[left:right]
            if len(seg) < 2:
                break
            E_local = poisson_efield_1d(seg, eps_seg, dx_m, E0=0.0)
            psi = -np.trapezoid(E_local, dx=dx_m)
            if abs(psi) >= (V_bi - V_app):
                break
            if lo > 0: lo -= 1
            if hi < n - 1: hi += 1
            if lo <= 0 and hi >= n-1: break
        mask[max(lo,0):min(hi+1,n)] = True
    return mask

def collection_probability_1d(depl_mask, L_nm, dx_nm, E_field=None, Vt=Vt_default):
    d_to_depl = distance_transform_edt(~depl_mask) * dx_nm
    eta = np.where(depl_mask, 1.0, np.exp(-d_to_depl / np.clip(L_nm, 1.0, None)))
    if E_field is not None:
        EL_over_Vt = np.abs(E_field) * (L_nm * 1e-9) / Vt
        with np.errstate(invalid="ignore", divide="ignore"):
            eta_drift = np.where(EL_over_Vt > 1e-6,
                                 (1.0 - np.exp(-EL_over_Vt)) / EL_over_Vt,
                                 0.0)
        eta = np.clip(eta + (1 - eta) * eta_drift, 0.0, 1.0)
    return eta

H, W = Nnet_map.shape
eps_r_map = np.full((H, W), 11.7)

L_map = np.full((H, W), 1e-3)
N_mask = (Nnet_map > 0) & SEMI_MASK
P_mask = (Nnet_map < 0) & SEMI_MASK
L_map[N_mask] = diffusion_length(Nnet_map[N_mask], "hole")
L_map[P_mask] = diffusion_length(-Nnet_map[P_mask], "electron")

E_map    = np.zeros((H, W))
depl_map = np.zeros((H, W), dtype=bool)
junc_points = []
dx_m = dx_nm * 1e-9

for x in range(W):
    col_semi = SEMI_MASK[:, x]
    if not col_semi.any():
        continue
    rows = np.where(col_semi)[0]
    r0, r1 = rows.min(), rows.max() + 1
    Nnet = Nnet_map[r0:r1, x]
    eps_c = eps_r_map[r0:r1, x]
    E_line = poisson_efield_1d(Nnet, eps_c, dx_m, E0=0.0)
    E_map[r0:r1, x] = E_line
    junctions = locate_junctions(Nnet)
    for j in junctions:
        junc_points.append((r0 + j, x))
    depl_map[r0:r1, x] = depletion_mask_1d(Nnet, eps_c, dx_m, junctions, V_bi=0.7)

E_map = np.where(SEMI_MASK, E_map, 0.0)
COLLECT = collection_probability_1d(depl_map, L_map, dx_nm, E_field=E_map)
COLLECT = np.where(SEMI_MASK, COLLECT, 0.0)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
signed = np.where(SEMI_MASK, np.sign(Nnet_map)*np.log10(np.clip(np.abs(Nnet_map),1,None)), 0)
im = axes[0,0].imshow(signed, cmap="bwr", vmin=-18, vmax=18)
axes[0,0].set_title("sign(N_net) log10|N_net|"); plt.colorbar(im, ax=axes[0,0])
im = axes[0,1].imshow(np.where(SEMI_MASK, E_map, np.nan), cmap="PuOr")
axes[0,1].set_title("E_y [V/m]"); plt.colorbar(im, ax=axes[0,1])
for (yy, xx) in junc_points[::5]:
    axes[0,1].plot(xx, yy, "kx", ms=3)
axes[1,0].imshow(IMG_RGB); axes[1,0].imshow(depl_map, cmap="Reds", alpha=0.45)
axes[1,0].set_title("depletion mask"); axes[1,0].set_axis_off()
im = axes[1,1].imshow(np.where(SEMI_MASK, COLLECT, np.nan), cmap="magma", vmin=0, vmax=1)
axes[1,1].set_title("collection probability"); plt.colorbar(im, ax=axes[1,1])
plt.tight_layout(); plt.show()
print(f"junctions found: {len(junc_points)}")


## 9. Circuit terminals and contact classification

Terminals are `{role, region_id, bias_V}` with role in
`{ammeter_gnd, voltage, ground}`. For each metal terminal, we look at
neighboring regions and classify each contact:

- Metal on n-type Si: Schottky if $\Phi_m > \Phi_s$, Ohmic if $\Phi_m \le \Phi_s$
- Metal on p-type Si: Schottky if $\Phi_m < \Phi_s$, Ohmic if $\Phi_m \ge \Phi_s$

Schottky barrier: $\Phi_m - \chi$ (n-type) or $E_g - (\Phi_m - \chi)$ (p-type).

In [ ]:
def contact_type(metal_name, semi_name, semi_doping):
    metal = MATERIALS.get(metal_name); semi = MATERIALS.get(semi_name)
    if metal is None or semi is None:
        return "unknown", None
    Phi_m = float(metal.work_function())
    Phi_s = float(semi.work_function(N=semi_doping))
    chi   = semi.electron_affinity_eV
    Eg    = float(semi.bandgap(N=semi_doping))
    if semi.doping_type == "N":
        kind = "Schottky" if Phi_m > Phi_s else "Ohmic"
        phi_B = Phi_m - chi
    else:
        kind = "Schottky" if Phi_m < Phi_s else "Ohmic"
        phi_B = Eg - (Phi_m - chi)
    return kind, float(phi_B)

def neighbors_of_region(regions, rid):
    mask = regions == rid
    dil = binary_dilation(mask) & ~mask
    return np.unique(regions[dil])

TERMINALS = []
for term in CONFIG["circuit"]:
    rid = term["region_id"]
    mat_name = CONFIG["region_assignment"].get(rid)
    info = dict(term, material=mat_name, contacts=[])
    if mat_name and MATERIALS.get(mat_name, Material("","")).kind == "Metal":
        for nrid in neighbors_of_region(REGIONS, rid):
            nrid = int(nrid)
            if nrid in (0, rid): continue
            n_mat = CONFIG["region_assignment"].get(nrid)
            if not n_mat: continue
            n_mask = (REGIONS == nrid) & SEMI_MASK
            if not n_mask.any(): continue
            N_here = float(np.mean(np.abs(Nnet_map[n_mask])))
            if n_mat == "SIMS":
                eff_name = "N-type_Si" if Nnet_map[n_mask].mean() > 0 else "P-type_Si"
            else:
                eff_name = n_mat
            kind, phi_B = contact_type(mat_name, eff_name, N_here)
            info["contacts"].append(
                dict(neighbor_id=nrid, semi=eff_name, kind=kind, phi_B_eV=phi_B))
    TERMINALS.append(info)

print("Circuit terminals:")
for t in TERMINALS:
    print(" ", t)


## 10. Band diagram along the active contact-to-contact path

Pick a 1-D path between the first two terminals (or the sample midline if
none configured). Vacuum, conduction, valence and Fermi levels are
computed from the local material + Poisson-integrated potential + applied
bias.

In [ ]:
def sample_line(y0, x0, y1, x1, n=400):
    ys = np.linspace(y0, y1, n).astype(int)
    xs = np.linspace(x0, x1, n).astype(int)
    return ys, xs

def choose_path(terminals, regions):
    if len(terminals) >= 2:
        c0 = measure.regionprops((regions == terminals[0]["region_id"]).astype(int))
        c1 = measure.regionprops((regions == terminals[1]["region_id"]).astype(int))
        if c0 and c1:
            return (int(c0[0].centroid[0]), int(c0[0].centroid[1]),
                    int(c1[0].centroid[0]), int(c1[0].centroid[1]))
    H, W = regions.shape
    return (0, W // 2, H - 1, W // 2)

def band_diagram_along(path, N_net, eps_r, eff_mat, materials, dx_nm, V_app=0.0):
    y0, x0, y1, x1 = path
    ys, xs = sample_line(y0, x0, y1, x1)
    Nnet_line = N_net[ys, xs]
    eps_line  = eps_r[ys, xs]
    dl_nm = dx_nm * math.hypot(y1-y0, x1-x0) / max(len(ys)-1, 1)
    E_line = poisson_efield_1d(Nnet_line, eps_line, dl_nm*1e-9)
    psi = -cumulative_trapezoid(E_line, dx=dl_nm*1e-9, initial=0.0)
    psi = psi + V_app * np.linspace(0, 1, len(ys))
    Ec = np.zeros_like(psi); Ev = np.zeros_like(psi)
    Evac = np.zeros_like(psi); EF = np.zeros_like(psi)
    for i, (yy, xx) in enumerate(zip(ys, xs)):
        name = eff_mat[yy, xx] or "Si"
        mat = materials.get(name)
        Evac[i] = -psi[i]
        if mat is None or mat.kind != "Semi":
            wf = float(mat.work_function()) if mat else 4.5
            EF[i] = Evac[i] - wf
            Ec[i] = EF[i]; Ev[i] = EF[i]
            continue
        chi = mat.electron_affinity_eV
        N   = abs(Nnet_line[i])
        Eg  = float(mat.bandgap(N=N))
        Ec[i] = Evac[i] - chi
        Ev[i] = Ec[i] - Eg
        EF[i] = Ec[i] - Eg / 2
    return dl_nm, np.arange(len(ys))*dl_nm, Evac, Ec, Ev, EF, Nnet_line

path = choose_path(TERMINALS, REGIONS)
V_bias = 0.0
for t in TERMINALS:
    if t["role"] == "voltage":
        V_bias = t.get("bias_V", 0.0); break
dl, ll, Evac, Ec, Ev, EF, Nline = band_diagram_along(
    path, Nnet_map, np.full_like(Nnet_map, 11.7, dtype=float),
    EFF_MAT, MATERIALS, dx_nm, V_app=V_bias,
)

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(ll, Evac, label="E_vac"); axes[0].plot(ll, Ec, label="E_c")
axes[0].plot(ll, Ev, label="E_v"); axes[0].plot(ll, EF, "k--", label="E_F")
axes[0].set_ylabel("energy [eV]"); axes[0].legend()
axes[0].set_title(f"Band diagram (bias = {V_bias} V)")
axes[1].semilogy(ll, np.abs(Nline)+1e10)
axes[1].set_xlabel("path [nm]"); axes[1].set_ylabel("|N_net| [cm$^{-3}$]")
plt.tight_layout(); plt.show()


## 11. EBIC and SEEBIC 2-D maps

$I_{EBIC}(r_b) = q \cdot G_{tot} \cdot (\eta_{collect} * K_{gen})(r_b)$

SEEBIC uses a mean-depth surface-escape factor
$\bar{P}_{esc} = (1 - e^{-t/\lambda_{SE}}) \cdot \lambda_{SE}/t$
applied to the generation kernel convolution over the sample mask.

In [ ]:
def convolve_map(field, kernel):
    return fftconvolve(field, kernel, mode="same")

EBIC_map = q * G_TOT * convolve_map(COLLECT, G_KERNEL)
lam = CONFIG["beam"]["secondary_escape_nm"]
t   = CONFIG["thickness_nm"]
P_esc_mean = (1.0 - math.exp(-t/max(lam,1e-6))) * (lam / max(t,1e-6))
SEEBIC_map = q * G_TOT * P_esc_mean * convolve_map(
    np.where(SEMI_MASK, 1.0, 0.0), G_KERNEL)

ebic_show   = np.where(SEMI_MASK, EBIC_map,   np.nan)
seebic_show = np.where(SEMI_MASK, SEEBIC_map, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(ebic_show, cmap="viridis")
axes[0].set_title("EBIC [A]"); plt.colorbar(im, ax=axes[0])
im = axes[1].imshow(seebic_show, cmap="magma")
axes[1].set_title(f"SEEBIC [A] (lambda_SE={lam} nm)"); plt.colorbar(im, ax=axes[1])
for ax in axes: ax.set_axis_off()
plt.tight_layout(); plt.show()
print(f"EBIC peak = {np.nanmax(ebic_show):.3e} A;  "
      f"SEEBIC peak = {np.nanmax(seebic_show):.3e} A")


## 12. 6-panel dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].imshow(IMG_RGB); axes[0,0].set_title("input"); axes[0,0].set_axis_off()

overlay = color.label2rgb(REGIONS, IMG_RGB, alpha=0.4, bg_label=0)
axes[0,1].imshow(overlay); axes[0,1].set_title("regions"); axes[0,1].set_axis_off()

signed = np.where(SEMI_MASK, np.sign(Nnet_map)*np.log10(np.clip(np.abs(Nnet_map),1,None)), np.nan)
im = axes[0,2].imshow(signed, cmap="bwr", vmin=-18, vmax=18)
axes[0,2].set_title("sign(N_net) log10|N_net|"); plt.colorbar(im, ax=axes[0,2], fraction=0.045)
axes[0,2].set_axis_off()

im = axes[1,0].imshow(np.where(SEMI_MASK, np.abs(E_map), np.nan), cmap="PuOr")
axes[1,0].set_title("|E| [V/m]"); plt.colorbar(im, ax=axes[1,0], fraction=0.045)
axes[1,0].set_axis_off()

axes[1,1].imshow(IMG_RGB); axes[1,1].imshow(depl_map, cmap="Reds", alpha=0.45)
axes[1,1].set_title("depletion"); axes[1,1].set_axis_off()

im = axes[1,2].imshow(np.where(SEMI_MASK, EBIC_map, np.nan), cmap="viridis")
axes[1,2].set_title("EBIC [A]"); plt.colorbar(im, ax=axes[1,2], fraction=0.045)
axes[1,2].set_axis_off()

plt.tight_layout(); plt.show()


## 13. Sanity tests

In [ ]:
# Arora mobility check
mu_e = arora_mobility(1e15, "electron")
print(f"mu_e(1e15) = {mu_e:.1f} cm^2/V/s  (~1350 expected)")

# BGN checks
print(f"BGN(1e17) = {bgn_delta_eV(np.array([1e17]))[0]*1000:.1f} meV  (expect 0)")
print(f"BGN(1e19) = {bgn_delta_eV(np.array([1e19]))[0]*1000:.1f} meV  (~90 meV)")

# V_bi for abrupt PN with Na=Nd=1e17
Vt = kB * T_default / q
Nc = MATERIALS["Si"].Nc(); Nv = MATERIALS["Si"].Nv()
ni = math.sqrt(Nc*Nv) * math.exp(-1.12/2/Vt)
Vbi_ref = Vt * math.log(1e17 * 1e17 / (ni*ni))
print(f"V_bi (Na=Nd=1e17 Si) = {Vbi_ref:.3f} V  (~0.7 V)")

# Contact type
kind, phi = contact_type("Al", "N-type_Si", 1e17)
print(f"Al on N-Si (1e17): {kind}, phi_B = {phi}")
kind, phi = contact_type("Pt", "N-type_Si", 1e15)
print(f"Pt on N-Si (1e15): {kind}, phi_B = {phi}")
